# 프로젝트 1. 영화 리뷰 감정 분석 (torchtext 없는 버전)

**torchtext 의존성 제거** — HuggingFace `datasets` + 표준 Python으로 대체

| 기존 (torchtext) | 대체 |
|---|---|
| `torchtext.datasets.IMDB` | `datasets.load_dataset('imdb')` |
| `get_tokenizer('basic_english')` | regex 기반 함수 |
| `build_vocab_from_iterator` | `collections.Counter` |

In [1]:
# datasets는 Colab에 기본 설치되어 있음. 없으면 아래 주석 해제
# !pip install datasets

In [2]:
import os  # 운영체제 모듈 임포트
import re  # 정규표현식 모듈 임포트
import torch  # 파이토치 임포트
import torch.nn as nn  # 신경망 모듈 임포트
import torch.nn.functional as F  # 신경망 함수 모듈 임포트
from collections import Counter  # 빈도수 계산을 위한 Counter 임포트
from datasets import load_dataset  # 허깅페이스 데이터셋 로드 함수 임포트
from torch.utils.data import Dataset, DataLoader  # 데이터셋/데이터로더 유틸리티 임포트
from torch.nn.utils.rnn import pad_sequence  # 시퀀스 패딩 유틸리티 임포트

In [3]:
# 하이퍼파라미터
BATCH_SIZE = 64  # 배치 크기 설정
lr = 0.001  # 학습률 설정
EPOCHS = 10  # 에포크 횟수 설정
MIN_FREQ = 5       # 어휘 사전에 포함할 최소 빈도
MAX_LEN  = 256     # 텍스트 최대 길이

USE_CUDA = torch.cuda.is_available()  # GPU 사용 가능 여부
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")  # 기기 설정
print("다음 기기로 학습합니다:", DEVICE)  # 출력

In [4]:
# 토크나이저 — torchtext 없이 regex로 구현
def tokenizer(text):  # 토크나이저 함수 정의
    text = text.lower()  # 소문자 변환
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # 특수문자 제거
    return text.split()  # 공백 기준 분리

In [5]:
print("데이터 로딩중...")  # 로딩 메시지
try:  # 로드 시도
    raw_dataset = load_dataset('stanfordnlp/imdb')  # IMDB 데이터셋 로드
except Exception as e:  # 실패 시
    print(f"HF Hub 로드 실패 ({type(e).__name__}): {e}")  # 에러 출력
    print("Stanford 원본에서 직접 다운로드 시도...")  # 대체 방법
    import urllib.request, tarfile, glob
    URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
    TAR = "./aclImdb_v1.tar.gz"
    DIR = "./aclImdb"
    if not os.path.isdir(DIR):
        urllib.request.urlretrieve(URL, TAR)
        with tarfile.open(TAR) as tf:
            tf.extractall(".")

    def _load_split(split):  # 분할 로드 함수
        texts, labels = [], []
        for lbl_name, lbl_id in [('neg', 0), ('pos', 1)]:
            for path in glob.glob(f"{DIR}/{split}/{lbl_name}/*.txt"):
                with open(path, encoding='utf-8') as f:
                    texts.append(f.read())
                labels.append(lbl_id)
        return {'text': texts, 'label': labels}

    from datasets import Dataset as HFDataset, DatasetDict
    raw_dataset = DatasetDict({
        'train': HFDataset.from_dict(_load_split('train')),
        'test':  HFDataset.from_dict(_load_split('test')),
    })

print(f"train: {len(raw_dataset['train'])}개, test: {len(raw_dataset['test'])}개")

In [6]:
# 학습 데이터로 어휘 사전 구축 (Counter 사용)
print("어휘 사전 구축중...")  # 진행 메시지
counter = Counter()  # 카운터 객체 생성
for item in raw_dataset['train']:  # 학습 데이터 반복
    counter.update(tokenizer(item['text']))  # 토큰 빈도 업데이트

# 특수 토큰: <unk>=0 (미등록어), <pad>=1 (패딩)
SPECIALS = ['<unk>', '<pad>']  # 특수 토큰 정의
vocab_words = SPECIALS + [w for w, cnt in counter.most_common() if cnt >= MIN_FREQ]  # 사전 구성
word2idx = {word: idx for idx, word in enumerate(vocab_words)}  # 단어-인덱스 매핑

UNK_IDX = word2idx['<unk>']  # UNK 인덱스 저장
PAD_IDX = word2idx['<pad>']  # PAD 인덱스 저장
vocab_size = len(word2idx)  # 사전 크기 저장
print(f"어휘 사전 크기: {vocab_size}")  # 출력

In [7]:
def text_to_indices(text):  # 텍스트를 인덱스로 변환하는 함수
    tokens = tokenizer(text)[:MAX_LEN]   # 토큰화 및 길이 제한
    return [word2idx.get(tok, UNK_IDX) for tok in tokens]  # 인덱스 리스트 반환

In [8]:
# PyTorch Dataset 클래스
class IMDBDataset(Dataset):  # IMDB 데이터셋 클래스 정의
    def __init__(self, hf_dataset):  # 초기화
        self.data = hf_dataset  # 데이터 저장

    def __len__(self):  # 길이 반환
        return len(self.data)

    def __getitem__(self, idx):  # 아이템 반환
        item = self.data[idx]  # 해당 인덱스 데이터 획득
        text_ids = torch.tensor(text_to_indices(item['text']), dtype=torch.long)  # 텍스트 인덱스화
        label = torch.tensor(item['label'], dtype=torch.long)  # 레이블 텐서화
        return text_ids, label  # 반환


# 배치 내 길이가 다른 시퀀스를 PAD로 정렬
def collate_fn(batch):  # 배치 처리 함수 정의
    texts, labels = zip(*batch)  # 데이터 분리
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)  # 패딩 처리
    return texts_padded, torch.stack(labels)  # 텐서 결합 후 반환

In [9]:
# train 25000개를 80/20으로 나눠 train/val 생성
train_raw = raw_dataset['train'].shuffle(seed=42)  # 데이터 셔플
test_raw  = raw_dataset['test']  # 테스트 데이터

split_idx  = int(len(train_raw) * 0.8)   # 분할 지점 계산 (20000)
train_data = train_raw.select(range(split_idx))  # 학습 데이터 선택
val_data   = train_raw.select(range(split_idx, len(train_raw)))  # 검증 데이터 선택

trainset = IMDBDataset(train_data)  # 학습 데이터셋 인스턴스
valset   = IMDBDataset(val_data)  # 검증 데이터셋 인스턴스
testset  = IMDBDataset(test_raw)  # 테스트 데이터셋 인스턴스

train_iter = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)  # 학습 로더
val_iter   = DataLoader(valset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)  # 검증 로더
test_iter  = DataLoader(testset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)  # 테스트 로더

n_classes = 2  # 클래스 수
print("[학습셋]: %d [검증셋]: %d [테스트셋]: %d [단어수]: %d [클래스] %d"
      % (len(trainset), len(valset), len(testset), vocab_size, n_classes))  # 정보 출력

In [10]:
# 원본과 동일한 BasicGRU 모델
class BasicGRU(nn.Module):  # GRU 모델 정의
    def __init__(self, n_layers, hidden_dim, n_vocab, embed_dim, n_classes, dropout_p=0.2):  # 초기화
        super(BasicGRU, self).__init__()  # 부모 초기화
        print("Building Basic GRU model...")  # 생성 메시지
        self.n_layers = n_layers  # 레이어 수
        self.embed = nn.Embedding(n_vocab, embed_dim, padding_idx=PAD_IDX)  # 임베딩
        self.hidden_dim = hidden_dim  # 은닉 차원
        self.dropout = nn.Dropout(dropout_p)  # 드롭아웃
        self.gru = nn.GRU(embed_dim, self.hidden_dim, num_layers=self.n_layers, batch_first=True)  # GRU
        self.out = nn.Linear(self.hidden_dim, n_classes)  # 출력층

    def forward(self, x):  # 순전파
        x = self.embed(x)  # 임베딩 적용
        x = self.dropout(x)  # 드롭아웃 적용
        _, h_n = self.gru(x)  # GRU 실행 (은닉 상태 획득)
        h_n = h_n[-1]  # 마지막 레이어 상태 선택
        logit = self.out(h_n)  # 선형 결합
        return logit  # 결과 반환

In [11]:
def train(model, optimizer, train_iter):  # 학습 함수 정의
    model.train()  # 학습 모드 설정
    total_loss = 0  # 누적 손실
    for x, y in train_iter:  # 데이터 반복
        x, y = x.to(DEVICE), y.to(DEVICE)  # 기기로 이동
        optimizer.zero_grad()  # 기울기 초기화
        logit = model(x)  # 모델 실행
        loss = F.cross_entropy(logit, y)  # 손실 계산
        loss.backward()  # 역전파
        optimizer.step()  # 업데이트
        total_loss += loss.item()  # 손실 추가
    return total_loss / len(train_iter)  # 평균 손실 반환


def evaluate(model, data_iter):  # 평가 함수 정의
    model.eval()  # 평가 모드 설정
    corrects, total_loss = 0, 0  # 초기값 설정
    with torch.no_grad():  # 기울기 계산 중지
        for x, y in data_iter:  # 데이터 반복
            x, y = x.to(DEVICE), y.to(DEVICE)  # 기기 이동
            logit = model(x)  # 실행
            loss = F.cross_entropy(logit, y, reduction='sum')  # 손실 계산
            total_loss += loss.item()  # 손실 누적
            corrects += (logit.max(1)[1] == y).sum().item()  # 맞춘 개수 카운트
    size = len(data_iter.dataset)  # 전체 크기
    avg_loss = total_loss / size  # 평균 손실
    avg_accuracy = 100.0 * corrects / size  # 정확도 계산
    return avg_loss, avg_accuracy  # 반환

In [12]:
model = BasicGRU(1, 256, vocab_size, 128, n_classes, 0.5).to(DEVICE)  # 모델 인스턴스 및 이동
optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # 최적화 도구
print(model)  # 모델 출력

In [13]:
train_losses, val_losses, val_accuracies = [], [], []  # 기록용 리스트

best_val_loss = None  # 최고 검증 손실 초기화
for e in range(1, EPOCHS + 1):  # 에포크 반복
    train_loss = train(model, optimizer, train_iter)  # 학습 실행
    val_loss, val_accuracy = evaluate(model, val_iter)  # 검증 실행

    train_losses.append(train_loss)  # 기록
    val_losses.append(val_loss)  # 기록
    val_accuracies.append(val_accuracy)  # 기록

    print("[이폭: %d] 학습 오차:%5.2f | 검증 오차:%5.2f | 검증 정확도:%5.2f"
          % (e, train_loss, val_loss, val_accuracy))  # 현황 출력

    if not best_val_loss or val_loss < best_val_loss:  # 성능 개선 시 모델 저장
        if not os.path.isdir("snapshot"):  # 디렉토리 확인
            os.makedirs("snapshot")
        torch.save(model.state_dict(), './snapshot/txtclassification.pt')  # 저장
        best_val_loss = val_loss

In [14]:
model.load_state_dict(torch.load('./snapshot/txtclassification.pt'))  # 최적 모델 로드
test_loss, test_acc = evaluate(model, test_iter)  # 최종 테스트
print('테스트 오차: %5.2f | 테스트 정확도: %5.2f' % (test_loss, test_acc))  # 결과 출력

In [15]:
import matplotlib.pyplot as plt  # 시각화용 임포트

epochs = range(1, len(train_losses) + 1)  # 에포크 범위

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))  # 서브플롯 생성

ax1.plot(epochs, train_losses, 'b-o', label='Train Loss')  # 학습 손실 플롯
ax1.plot(epochs, val_losses,   'r-o', label='Val Loss')  # 검증 손실 플롯
ax1.set_title('Learning Curve — Loss', fontsize=14)  # 제목
ax1.set_xlabel('Epoch')  # x축 이름
ax1.set_ylabel('Loss')  # y축 이름
ax1.legend()  # 범례
ax1.grid(True)  # 격자

best_epoch = val_losses.index(min(val_losses)) + 1  # 최적 에포크 확인
ax1.axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best Model (Epoch {best_epoch})')  # 수직선 표시
ax1.legend()  # 범례 업데이트

ax2.plot(epochs, val_accuracies, 'g-o', label='Val Accuracy')  # 정확도 플롯
ax2.set_title('Validation Accuracy', fontsize=14)  # 제목
ax2.set_xlabel('Epoch')  # x축
ax2.set_ylabel('Accuracy (%)')  # y축
ax2.legend()  # 범례
ax2.grid(True)  # 격자
ax2.axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.7)  # 수직선

plt.tight_layout()  # 레이아웃 정렬
plt.savefig('learning_curve.png', dpi=120)  # 저장
plt.show()  # 출력
print(f"\nBest model: Epoch {best_epoch} | Val Loss {min(val_losses):.4f}")